# Roteiro de Execução — Pipeline de Viagens a Serviço

**Autor:** Nivaldo Rodrigues — Projeto avaliativo, Análise de Dados com Python, Módulo 1, Semana 13.

Este notebook é um **roteiro para rodar o meu projeto** (Python + PostgreSQL, Arquitetura Medallion),
clicando célula por célula. As credenciais só se editam no Passo 2. O pipeline em si — schema SQL,
extração, transformação e análise — está nos arquivos `0_criar_banco.sql`, `1_extrair.py`,
`2_transformar.py` e `3_analise.ipynb`, que são as entregas avaliadas.

| Passo | O que acontece | Arquivo do projeto |
|---|---|---|
| 1 | instala as bibliotecas | `requirements.txt` |
| 2 | grava a senha no `.env` e testa a conexão | `.env.example` → `.env` |
| 3 | cria o banco `transparencia` e as 8 tabelas | `0_criar_banco.sql` |
| 4 | baixa os dados e carrega a camada Raw | `1_extrair.py` |
| 5 | converte tipos e carrega a camada Silver | `2_transformar.py` |
| 6 | cria a camada Gold e responde as 7 perguntas | `3_analise.ipynb` |
| 7 | confere todas as tabelas | — |

**Onde executar:** no VS Code, na minha máquina — o projeto precisa do PostgreSQL local, então não roda
diretamente no Google Colab.


## Passo 1 — Instalar as bibliotecas do projeto

In [ ]:
%pip install -r requirements.txt


**Deu certo?** Deve aparecer `Successfully installed ...` ou `Requirement already satisfied ...`. Se instalou algo agora, reinicie o kernel do notebook antes de continuar.

## Passo 2 — Senha do PostgreSQL no arquivo `.env`

A senha nunca fica escrita nos scripts: fica só no `.env` (protegido pelo `.gitignore`). A célula
abaixo lê o modelo `.env.example`, troca o texto `sua_senha_aqui` pela minha senha real e grava o
resultado em `.env`.

**A única linha que eu edito:** troque `SUA_SENHA_AQUI` pela senha do meu PostgreSQL, mantendo as aspas.


In [ ]:
SENHA = "SUA_SENHA_AQUI"

modelo = open(".env.example", encoding="utf-8").read()
conteudo = modelo.replace("sua_senha_aqui", SENHA)

with open(".env", "w", encoding="utf-8") as arquivo:
    arquivo.write(conteudo)

print("Arquivo .env gravado.")


Testando a conexão antes de seguir:

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(host="127.0.0.1", port=5432, dbname="postgres",
                                user="postgres", password=SENHA)
    print("Conexão OK. O PostgreSQL aceitou a senha.")
    conexao.close()
except Exception as erro:
    print("Não foi possível conectar.")
    print("Tipo do erro:", type(erro).__name__)
    print("Mensagem:", erro)


**Apareceu `Conexão OK`?** Siga para o Passo 3.

**Apareceu erro?**

| Erro | Causa provável | O que fazer |
|---|---|---|
| `UnicodeDecodeError` | senha errada — o Postgres em português responde com acentos que o Python não decodifica direito | corrija a senha no Passo 2 e rode de novo |
| `OperationalError: Connection refused` | o serviço do PostgreSQL não está rodando | abra 'Serviços' no Windows, procure `postgresql-x64-18` e clique em Iniciar |


## Passo 3 — Fase 0: banco e tabelas (`0_criar_banco.sql`)

O arquivo tem duas partes, porque `CREATE DATABASE` não pode rodar dentro da mesma transação de quem
vai usar o banco: a **Parte 1** cria o banco `transparencia` (conectado no banco `postgres`, com
`autocommit = True`); a **Parte 2** cria as 4 tabelas Raw e as 4 Silver (com `PRIMARY KEY`,
`FOREIGN KEY`, `NOT NULL`, `CHECK` e `UNIQUE`), já conectada no banco do projeto.

A célula corta o arquivo `.sql` no marcador `PARTE 2` (`.split("PARTE 2")`) pra rodar cada parte na
conexão certa.


In [ ]:
import psycopg2

from banco import conectar
from config import POSTGRES_CONFIG

NOME_BANCO = POSTGRES_CONFIG["dbname"]

# Parte 1: cria o banco (conectado no banco "postgres")
conexao = psycopg2.connect(host=POSTGRES_CONFIG["host"], port=POSTGRES_CONFIG["port"], dbname="postgres",
                            user=POSTGRES_CONFIG["user"], password=POSTGRES_CONFIG["password"])
conexao.autocommit = True
cursor = conexao.cursor()
cursor.execute("SELECT 1 FROM pg_database WHERE datname = %s", (NOME_BANCO,))
if cursor.fetchone() is None:
    cursor.execute(f"CREATE DATABASE {NOME_BANCO}")
    print(f"Banco '{NOME_BANCO}' criado.")
else:
    print(f"O banco '{NOME_BANCO}' já existia.")
conexao.close()

# Parte 2: cria as 8 tabelas (conectado no banco do projeto)
texto_sql = open("0_criar_banco.sql", encoding="utf-8").read()
parte_2 = texto_sql.split("PARTE 2")[-1]
parte_2 = parte_2.split("\n", 1)[1]  # descarta o resto da linha do marcador

conexao = conectar()
cursor = conexao.cursor()
cursor.execute(parte_2)
conexao.commit()

cursor.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public' ORDER BY table_name
""")
tabelas = [linha[0] for linha in cursor.fetchall()]
conexao.close()
print(f"{len(tabelas)} tabelas no banco:", tabelas)


**Resultado esperado:** `8 tabelas no banco`, 4 com prefixo `raw_` e 4 com prefixo `silver_`, todas ainda vazias.

## Passo 4 — Fase 1: download e camada Raw (`1_extrair.py`)

`%run 1_extrair.py` executa meu script inteiro aqui no notebook. Ele baixa o `.zip` do Google Drive
com `gdown`, extrai os 4 CSVs em `data/`, lê cada um em blocos de 50 mil linhas (tudo como texto) e
carrega nas 4 tabelas Raw, truncando antes (idempotente) e com `try/except` em cada etapa (resiliente).


In [ ]:
%run 1_extrair.py


**Resultado esperado:** 341.860 viagens, 606.916 pagamentos, 167.260 passagens e 763.349 trechos carregados na Raw, sem nenhuma linha de ERRO.

## Passo 5 — Fase 2: Raw → Silver (`2_transformar.py`)

Minhas funções de conversão: `converter_decimal()` (texto com vírgula → `Decimal`), `converter_data()`
(texto `DD/MM/AAAA` → `date`) e `limpar_texto()` (tira espaços e transforma vazio em `None`). A
`silver_viagem` é carregada primeiro (as outras 3 tabelas têm `FOREIGN KEY` apontando pra ela), com
`valor_total` e `duracao_dias` calculados nessa etapa.


In [ ]:
%run 2_transformar.py


**Resultado esperado:** mensagem final `Pipeline Raw -> Silver concluído com sucesso.`, sem ERRO.

## Passo 6 — Fase 3: camada Gold e as 7 perguntas (`3_analise.ipynb`)

A análise fica num notebook próprio, porque é conteúdo para leitura: cada pergunta tem a consulta SQL,
a tabela e o gráfico.

1. No Explorer do VS Code, abra **`3_analise.ipynb`**.
2. Selecione o mesmo kernel deste notebook.
3. Clique em **Run All**.

Esse notebook cria a camada Gold: a tabela **`gold_resumo_orgao`** e a view **`vw_resumo_orgao`**
(agregadas com `JOIN` + `GROUP BY`), e responde as 7 perguntas de negócio do projeto.

**Depois de rodar o `3_analise.ipynb`, volte aqui e rode o Passo 7.**


## Passo 7 — Conferência final

Conta as linhas de cada tabela do projeto. `to_regclass()` confere se a tabela existe antes de contar,
pra não dar erro se o `3_analise.ipynb` ainda não tiver rodado.


In [ ]:
from banco import conectar

TABELAS_DO_PROJETO = [
    "raw_viagem", "raw_pagamento", "raw_passagem", "raw_trecho",
    "silver_viagem", "silver_pagamento", "silver_passagem", "silver_trecho",
    "gold_resumo_orgao", "vw_resumo_orgao",
]

conexao = conectar()
cursor = conexao.cursor()
for tabela in TABELAS_DO_PROJETO:
    cursor.execute("SELECT to_regclass(%s)", (tabela,))
    if cursor.fetchone()[0] is None:
        print(f"{tabela:<24} ainda não existe (execute o 3_analise.ipynb)")
    else:
        cursor.execute(f"SELECT COUNT(*) FROM {tabela}")
        print(f"{tabela:<24} {cursor.fetchone()[0]:>9} linhas")
conexao.close()


**Resultado esperado:** as 4 tabelas Raw e as 4 Silver com as quantidades do Passo 4/5, e
`gold_resumo_orgao` com a mesma quantidade de linhas da view `vw_resumo_orgao`.

### ✅ Projeto executado

Fase 0 (banco e tabelas), Fase 1 (download + Raw), Fase 2 (Silver tipada) e Fase 3 (Gold + perguntas +
gráficos) concluídas. O `README.md` desta pasta documenta as decisões do projeto, as respostas das
7 perguntas e as melhorias futuras.
